# KurdiSent — Category-Level Error Outlier

This notebook produces **one figure** that identifies the category with an anomalous
misclassification rate in the XLM-R sentiment baseline.

**How the rate is computed**
- *Errors per category*: taken from the out-of-fold misclassified export
  (`..._misclassified.csv`, 1 row per wrongly predicted sample).
- *Total per category*: taken from the original corpus (`KurdiSent.csv`).
- `error_rate = errors / total` per category.

> **Note on denominators.** The training notebook drops rows with an empty
> `surface` before cross-validation, so a handful of samples may not appear in the
> out-of-fold set. Category totals are in the hundreds–thousands, so this changes
> the rates by well under a tenth of a percent and does not affect the outlier.

## 1 · Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

ORIG_PATH = "KurdiSent.csv"
MIS_PATH  = "kusa_baseline_cv_inc_error_analysis_alternative_misclassified.csv"

def load_csv(path):
    """Load a CSV; if it isn't found, fall back to an interactive Colab upload."""
    import os
    if os.path.exists(path):
        return pd.read_csv(path, encoding="utf-8-sig")
    try:
        from google.colab import files
        print(f"'{path}' not found — please upload it:")
        up = files.upload()
        fname = list(up.keys())[0]
        return pd.read_csv(fname, encoding="utf-8-sig")
    except Exception as e:
        raise FileNotFoundError(
            f"Could not find or upload '{path}'. Set the path at the top of this cell."
        ) from e

orig = load_csv(ORIG_PATH)   # original corpus: needs a 'category' column
mis  = load_csv(MIS_PATH)    # misclassified export: needs a 'category' column
print("original rows:", len(orig), "| misclassified rows:", len(mis))

## 2 · Error rate per category

In [ ]:
totals = orig["category"].value_counts()
errors = mis["category"].value_counts()

cats = [c for c in totals.index if c in errors.index]
tbl = pd.DataFrame({
    "category": cats,
    "total":  [int(totals[c]) for c in cats],
    "errors": [int(errors[c]) for c in cats],
})
tbl["error_rate_%"] = (100 * tbl["errors"] / tbl["total"]).round(2)
tbl = tbl.sort_values("error_rate_%", ascending=False).reset_index(drop=True)

overall_rate = 100 * len(mis) / int(totals.sum())
outlier_cat  = tbl.loc[tbl["error_rate_%"].idxmax(), "category"]

print(tbl.to_string(index=False))
print(f"\nOverall error rate: {overall_rate:.1f}%")
print(f"Outlier category  : {outlier_cat}")

## 3 · The figure

In [ ]:
# Sort ascending so the highest bar sits at the top of the horizontal chart.
plot_df = tbl.sort_values("error_rate_%", ascending=True).reset_index(drop=True)
outlier_idx = plot_df["error_rate_%"].idxmax()

HIGHLIGHT = "#c62828"   # outlier
MUTED     = "#b9c2cc"   # everyone else
INK       = "#37474f"

fig, ax = plt.subplots(figsize=(9, 5))

colors = [HIGHLIGHT if i == outlier_idx else MUTED for i in range(len(plot_df))]
ax.barh(plot_df["category"].str.capitalize(), plot_df["error_rate_%"],
        color=colors, edgecolor="white", height=0.68)

# overall-rate reference line
ax.axvline(overall_rate, color=INK, ls="--", lw=1.3, zorder=0)
ax.text(overall_rate + 0.3, -0.55, f"Overall error rate: {overall_rate:.1f}%",
        color=INK, fontsize=9, va="center")

# value + sample-size labels
for i, (r, n) in enumerate(zip(plot_df["error_rate_%"], plot_df["total"])):
    is_out = (i == outlier_idx)
    ax.text(r + 0.4, i, f"{r:.1f}%   (n={n:,})", va="center", fontsize=9.5,
            color=HIGHLIGHT if is_out else "#546e7a",
            fontweight="bold" if is_out else "normal")

# label inside the outlier bar
ax.annotate("Outlier", xy=(plot_df["error_rate_%"].iloc[outlier_idx] - 3, outlier_idx),
            fontsize=11, fontweight="bold", color="white", va="center", ha="center")

ax.set_xlabel("Misclassification rate (%)", fontsize=11)
ax.set_title("Sentiment Misclassification Rate by Category\n"
             "XLM-R large · 5-fold stratified CV (out-of-fold)",
             fontsize=13, fontweight="bold", loc="left")
ax.set_xlim(0, plot_df["error_rate_%"].max() + 6)
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="x", ls=":", alpha=0.4)

plt.tight_layout()
plt.savefig("category_error_outlier.png", dpi=150, bbox_inches="tight")
plt.show()